# A Latent Diffusion model combining ideas from CS336's language model and a couple image generation papers

Based on:
* The language transformer from CS336 (multihead self attention, adamw, swiglu, rope)
* X-prediction and flow matching from [Let Denoising Generative Models Denoise (Tianhong Li, Kaiming He, et al., 2025)](https://arxiv.org/abs/2511.13720)
* A Vector Quantized latent space from ["Generative Refinement Networks for Visual Synthesis", 
(Han et al., 2026)](https://arxiv.org/abs/2604.13030)

* https://claude.ai/share/9c11840b-4ad9-4736-8119-47e2060b7a02 - GRN -> JiT
* https://claude.ai/chat/4b918b44-b26c-445f-ac9d-7899e60f6875 - Rope based autoencoder vs limited receptive field
* https://share.google/aimode/GQu5a6AvrHfCnCG3G - LPIPS, Euler-Maruyama sampling

In [ ]:
import torch
import hierarchical_binary_quantization.example_narrow_reciptive_field_autoencoder as enrfa
import hierarchical_binary_quantization.example_latent_diffusion_model as eldm
import hierarchical_binary_quantization.misc.checkpoint_helpers as ch
import hierarchical_binary_quantization.misc.dataset_helpers as dh
import hierarchical_binary_quantization.misc.flow_matching_diffusion_helpers as fmdh
import hierarchical_binary_quantization.misc.image_helpers as ih
from hierarchical_binary_quantization.misc.flow_matching_diffusion_helpers import generate_using_direct_x_prediction
from hierarchical_binary_quantization.misc.flow_matching_diffusion_helpers import generate_using_stochastic_heun
from hierarchical_binary_quantization.misc.flow_matching_diffusion_helpers import generate_using_sde
from hierarchical_binary_quantization.misc.flow_matching_diffusion_helpers import generate_simple, generate
import IPython.display as ipd

dataset_name,img_height, img_width = 'fantasy',512,384
grid_height, grid_width = img_height//8, img_width//8

batch_size = 2 # 6 worked for 256x256 before lpips :(
good_autoencoder_checkpoint = '../assets/reallygood_LocalQuantizingAutoencoder_2026-09-12_07-44-00_ps8,qd16,ld32,body:res,attn,res.pth'
good_latent_xpred_diffusion_cp = 'checkpoints/good/best_so_far_2026-10-03_MinimalSpatialViT_384x512_2026-10-03_08-23-41_3-dataset-autoencoder_cs336_latent_jit.pth.ema.pth'

device='cuda' if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device


In [ ]:
#checkpoint_path = '../../hierarchical-binary-quantization/notebooks/checkpoints/ExampleQuantizingAutoencoder_2026-08-15_12-48-36_.pth'
#autoencoder = hbq_misc.checkpoint_helpers.create_from_checkpoint(ExampleQuantizingAutoencoderWithRope, good_autoencoder_checkpoint)
autoencoder = ch.create_from_checkpoint(enrfa.LocalQuantizingAutoencoder,good_autoencoder_checkpoint)
autoencoder = autoencoder.eval().requires_grad_(False).to(device)

ldm = ch.create_from_checkpoint(eldm.MinimalSpatialViT,good_latent_xpred_diffusion_cp)
ldm = ldm.eval().requires_grad_(False).to(device)

pass

In [ ]:
seed = 0

In [ ]:
def compare_samplers(ldm,seed,steps=20):
    torch.manual_seed(seed)
    l1 = generate_simple(ldm,2,16,grid_width=grid_width,grid_height=grid_height,steps=steps,t_eps=0.01)
    torch.manual_seed(seed)
    l2 = generate_using_stochastic_heun(ldm,2,16,grid_width=grid_width,grid_height=grid_height,steps=steps,t_eps=0.01)
    torch.manual_seed(seed)
    t_schedule = torch.arange(0.0,1.0,0.1)
    l3 = generate_using_direct_x_prediction(ldm,torch.zeros(2,16,grid_height,grid_width),t_schedule=t_schedule)
    i1,i2,i3 = [autoencoder.decode(autoencoder.post_quant(l)) for l in [l1,l2,l3]]
    p1,p2,p3 = [ih.tensor_to_pil(i[0]) for i in [i1,i2,i3]]
    return ih.html_for_images([p1,p2,p3])


In [ ]:
h = compare_samplers(ldm=ldm,seed=4)
ipd.display(ipd.HTML(h))

In [ ]:
h = compare_samplers(ldm=ldm,seed=9)
ipd.display(ipd.HTML(h))

In [ ]:
for seed in range(10,20):
    h = compare_samplers(ldm=ldm,seed=seed)
    print(seed)
    ipd.display(ipd.HTML(h))

In [ ]:
for seed in range(50,100):
    h = compare_samplers(ldm=ldm,seed=seed)
    print(seed)
    ipd.display(ipd.HTML(h))

In [ ]:
1/0

# TODO - further explore these other loop ideas

In [ ]:
x.shape

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.rand(1,16,grid_height,grid_width).to(device)
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
for t_val in torch.arange(0.0,1.0,0.1):
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)

    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.rand(1,16,grid_height,grid_width).to(device)*2-1
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
for t_val in torch.arange(0.0,1.0,0.1):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)

    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.rand(1,16,grid_height, grid_width).to(device)*2-1
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x) * 1.1
for t_val in torch.arange(0.0,1.0,0.1):
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj._orig_mod(z, t)

    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.rand(1,16,grid_height, grid_width).to(device)*2-1
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e0w,e1w=0,1
e0 = torch.randn_like(x)
for t_val in torch.arange(0.0,1.0,0.1):
    e1 = torch.randn_like(x)
    e = (e0*e0w + e1*e1w) / (e0w+e1w)**0.5
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.rand(1,16,grid_height, grid_width).to(device)*2-1
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e0 = torch.randn_like(x) 
for t_val in torch.arange(0.0,1.0,0.1):
    e1 = torch.randn_like(x)
    e = (e0 + e1) / (1+1)**0.5
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
grid_height, grid_width

## Repeatedly noise and denoise at a given T

In [ ]:
# repeatedly noise and denoise at a given T


lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.975
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
# repeatedly noise and denoise at a given T


lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.75
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
# repeatedly noise and denoise at a given T


#lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
#x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.95
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
# repeatedly noise and denoise at a given T


lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.95
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
# repeatedly noise and denoise at a given T


lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.95
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
# repeatedly noise and denoise at a given T


lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.69
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
# repeatedly noise and denoise at a given T


lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.69
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))